----
<div style="display: flex; align-items: center;">
    <img src="https://developers.lseg.com/content/dam/devportal/icons/logo/lseg-logo.svg" width="20%" style="vertical-align: top;">
    <h1 style="margin-left: 20px;">Data Library for Python</h1>
</div>

----

## Content layer - Tradefeedr - Pre-Trade Forecast
This notebook demonstrates FX algorithmic pre-trade forecasts using `lseg.data.content.tradefeedr.PreTradeForecastDefinition`: a single scenario, multiple scenarios, field and model selection, input validation and asynchronous requests.

The tests call the Access layer function `ld.tradefeedr.get_fx_algo_pre_trade_forecast`. That function constructs the same Content layer definition and returns `response.data.df`. Here we use the Content layer directly to retain access to the response and raw data.

**Prerequisites:** a running LSEG Workspace desktop session configured for the Data Library, and entitlement to Tradefeedr. The tests use a desktop session. Requests send the scenario symbols, USD quantities and arrival times to the configured service; they do not place or execute orders. The historical timestamps and quantities below come from those tests. Data availability depends on your entitlements and the service; errors are not suppressed.

See [Parent Orders](EX-2.13.02-Tradefeedr-ParentOrders.ipynb) for retrieving existing order analytics.

#### Learn more

Visit the [LSEG Developer Portal](https://developers.lseg.com/en) for learning materials about the LSEG Data Library for Python.

#### Getting Help and Support

For questions about the API, visit the [LSEG Developer Community](https://community.developers.lseg.com/).

----

## Set the configuration file location
Run the notebook from its own directory. `LD_LIB_CONFIG_PATH` points to the project's `Configuration` directory. Configure the desktop session and application key in `lseg-data.config.json`, not in the notebook.

In [ ]:
import os
os.environ["LD_LIB_CONFIG_PATH"] = "../../../Configuration"

## Some Imports to start with
The public module name is `tradefeedr`.

In [ ]:
import asyncio

import lseg.data as ld
from lseg.data.content import tradefeedr

## Open the data session
`open_session()` selects the default session in `lseg-data.config.json`. Set that default to your configured desktop session to reproduce the test environment.

In [ ]:
ld.open_session()

## Request a single forecast
The scalar example from the tests uses `EURUSD`, a USD-equivalent order quantity of 50 million, the `tradefeedr/Global` model and the `CostOfLiquidity` field. Tradefeedr symbols such as `EURUSD` are not RICs such as `EUR=`.

Use the documented arrival-time format `YYYY-MM-DDTHH:MM:SS`. The equivalent space-separated timestamps in the tests are also parsed by the library. `order_quantity_usd` is expressed in USD, not in units of the pair's base currency.

In [ ]:
single_response = tradefeedr.PreTradeForecastDefinition(
    universe="EURUSD",
    order_quantity_usd=50e6,
    arrival_time="2023-08-10T11:52:00",
    fields="CostOfLiquidity",
    forecast_model="tradefeedr/Global",
).get_data()
single_response.data.df

## Request multiple scenarios
`universe`, `order_quantity_usd` and `arrival_time` are aligned by position: the first symbol uses the first quantity and first timestamp. Supply lists of equal length; a scalar is treated as one item and is not broadcast across a list.

The active test uses the four fields below. Omitting `forecast_model` requests the library's default model set: `tradefeedr/Global`, `tradefeedr/Fast` and `tradefeedr/Slow`. A scenario may therefore produce multiple result rows; do not assume one row per symbol.

In [ ]:
response = tradefeedr.PreTradeForecastDefinition(
    universe=["EURUSD", "USDJPY"],
    order_quantity_usd=[50e6, 30e6],
    arrival_time=["2023-08-10T11:52:00", "2023-08-10T12:30:00"],
    fields=["ArrivalTime", "Symbol", "OrderQuantityUSD", "CostOfLiquidity"],
).get_data()
response.data.df

### Inspect the raw response
`data.df` presents the result as a DataFrame; `data.raw` retains the service payload. The DataFrame builder converts `ArrivalTime` to datetime values using UTC and removes the timezone annotation. Avoid interpreting these timezone-naive values as local machine time.

In [ ]:
response.data.raw

## Compare selected forecast models
A model can be passed as a string or a list. Include `ForecastModel` in the selected fields to identify each result. Additional fields below are taken from the library's default forecast selection. If `fields` is omitted, the full default selection is used instead.

These are model outputs, not guaranteed execution outcomes or an instruction to trade.

In [ ]:
model_response = tradefeedr.PreTradeForecastDefinition(
    universe="EURUSD",
    order_quantity_usd=50e6,
    arrival_time="2023-08-10T11:52:00",
    fields=[
        "ArrivalTime", "Symbol", "OrderQuantityUSD", "ForecastModel",
        "CostOfLiquidity", "ForecastPerfBPS", "ForecastDurationMins", "ForecastRiskBPS",
    ],
    forecast_model=["tradefeedr/Global", "tradefeedr/Fast", "tradefeedr/Slow"],
).get_data()
model_response.data.df

## Validate scenario sizes locally
A mismatched number of symbols, quantities and times raises `PreTradeForecastInputSizeMismatch` when the definition is constructed. This intentionally invalid example makes no service request. Only the expected exception is caught.

In [ ]:
try:
    tradefeedr.PreTradeForecastDefinition(
        universe=["EURUSD", "USDJPY"],
        order_quantity_usd=[50e6],
        arrival_time=["2023-08-10T11:52:00", "2023-08-10T12:30:00"],
    )
except tradefeedr.PreTradeForecastInputSizeMismatch as error:
    print(f"Expected input validation error: {error}")
else:
    raise RuntimeError("Mismatched forecast scenario sizes were not rejected")

## Request forecasts asynchronously
The Content layer also exposes `get_data_async()`. This extends the synchronous test scenarios with concurrent independent requests using the same parameters. Jupyter supports top-level `await`; do not call `asyncio.run()` inside its running event loop. Request errors propagate to the cell.

In [ ]:
eur_response, jpy_response = await asyncio.gather(
    tradefeedr.PreTradeForecastDefinition(
        universe="EURUSD",
        order_quantity_usd=50e6,
        arrival_time="2023-08-10T11:52:00",
        fields=["Symbol", "ForecastModel", "CostOfLiquidity"],
        forecast_model="tradefeedr/Global",
    ).get_data_async(),
    tradefeedr.PreTradeForecastDefinition(
        universe="USDJPY",
        order_quantity_usd=30e6,
        arrival_time="2023-08-10T12:30:00",
        fields=["Symbol", "ForecastModel", "CostOfLiquidity"],
        forecast_model="tradefeedr/Global",
    ).get_data_async(),
)
display(eur_response.data.df)
display(jpy_response.data.df)

## Close the session
No orders were submitted and no local files were created. If an earlier request fails, run this cell to close the session.

In [ ]:
ld.close_session()